Step 3 test on the 1,747 recorder 5 clips. First check that the model files exist and that the allow-list names match the labels file exactly, because a mismatch would silently drop species.

In [1]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import sys, time, inspect
from pathlib import Path
import pandas as pd

HOME = Path.home()
CLIPS_DIR   = KWF_TEST_DIR / "split_clips"
OUTPUTS_DIR = KWF_TEST_DIR / "outputs"
MODELS_DIR = PIPELINE_DIR / "models"

sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

for f in ["birdnet_fp16_pruned493.onnx", "birdnet_fp16_pruned493.labels.txt", "geomodel_allowlist_218.txt"]:
    print(f"{f:40s} exists: {(MODELS_DIR / f).exists()}")
print("Clips available:", len(list(CLIPS_DIR.glob('*.wav'))))

birdnet_fp16_pruned493.onnx              exists: True
birdnet_fp16_pruned493.labels.txt        exists: True
geomodel_allowlist_218.txt               exists: True
Clips available: 1747


In [2]:
labels = [l.strip() for l in open(MODELS_DIR / "birdnet_fp16_pruned493.labels.txt")]
allow  = {l.strip() for l in open(MODELS_DIR / "geomodel_allowlist_218.txt") if l.strip()}

print("labels:", len(labels), "| allow-list:", len(allow))
print("found in labels:", len(allow & set(labels)), "| missing:", len(allow - set(labels)))
print("missing names:", sorted(allow - set(labels))[:10])

labels: 493 | allow-list: 218
found in labels: 218 | missing: 0
missing names: []


Run process_all_clips on the 1,747 recorder 5 clips using the same keyword arguments as run_pipeline.py step3_birdnet. Expected: 1,747 rows, columns clip_name / species / confidence, no “Error processing” messages.

In [3]:
from run_birdnet import process_all_clips

sig = inspect.signature(process_all_clips)
print("Signature:", sig)
print("Master-script args NOT accepted:", {"clips_folder", "output_csv"} - set(sig.parameters) or "none")

birdnet_csv = OUTPUTS_DIR / "nb03_birdnet_output.csv"
t0 = time.time()
process_all_clips(
    clips_folder=str(CLIPS_DIR),
    output_csv=str(birdnet_csv),
)
print(f"\nRuntime: {time.time() - t0:.1f}s")

Signature: (clips_folder, output_csv, confidence_threshold=0.25, use_allow_list=True)
Master-script args NOT accepted: none
Loading model: birdnet_fp16_pruned493.onnx
Model loaded. Labels: 493 species
Allow-list: 218 species
Processing 1,747 clips from /Users/huyguy/kwf_test/split_clips...


BirdNET: 100%|██████████| 1747/1747 [00:44<00:00, 39.04clip/s]


Done. 1,747 clips processed, 101 with detections.
Saved to: /Users/huyguy/kwf_test/outputs/nb03_birdnet_output.csv

Runtime: 44.9s


Validate the BirdNET output: row count, clip_name match with Step 1 and Step 2, list parsing, and whether detections respect the 0.25 threshold and the allow-list.

In [4]:
import ast

b = pd.read_csv(birdnet_csv)
print("Shape:", b.shape, "| columns:", b.columns.tolist())
print("Unique clip_names:", b["clip_name"].nunique())
print("Example row:\n", b.iloc[0].to_dict())
print("Raw type of 'species' cell:", type(b["species"].iloc[0]))

# 1. Do the names match Step 1 (files on disk) and Step 2 (Tiny CNN output)?
on_disk = {p.name for p in CLIPS_DIR.glob("*.wav")}
cnn = pd.read_csv(OUTPUTS_DIR / "nb02_recorder5_rerun_predictions.csv")
print("\nMatch with Step 1 files:", set(b["clip_name"]) == on_disk)
print("Match with Step 2 output:", set(b["clip_name"]) == set(cnn["clip_name"]))

# 2. Parse the list columns back into real lists
b["species_l"] = b["species"].apply(ast.literal_eval)
b["conf_l"]    = b["confidence"].apply(ast.literal_eval)
print("\nParsed OK. Lists equal length in every row:", (b["species_l"].str.len() == b["conf_l"].str.len()).all())

# 3. Do detections respect the rules?
allow = {l.strip() for l in open(MODELS_DIR / "geomodel_allowlist_218.txt") if l.strip()}
all_species = [s for lst in b["species_l"] for s in lst]
all_conf    = [c for lst in b["conf_l"] for c in lst]
print("Clips with detections:", (b["species_l"].str.len() > 0).sum())
print("Total detections:", len(all_species))
print("Species outside allow-list:", sum(s not in allow for s in all_species))
print("Min confidence:", min(all_conf), "| max:", max(all_conf))

# 4. What was detected?
print("\nTop species:")
print(pd.Series(all_species).value_counts().head(10))

Shape: (1747, 3) | columns: ['clip_name', 'species', 'confidence']
Unique clip_names: 1747
Example row:
 {'clip_name': 'Audio_Moth_5_20250319_163237.wav', 'species': '[]', 'confidence': '[]'}
Raw type of 'species' cell: <class 'str'>

Match with Step 1 files: True
Match with Step 2 output: True

Parsed OK. Lists equal length in every row: True
Clips with detections: 101
Total detections: 106
Species outside allow-list: 0
Min confidence: 0.2519 | max: 0.9451

Top species:
Myiothlypis fulvicauda_Buff-rumped Warbler         58
Myiornis atricapillus_Black-capped Pygmy-Tyrant    31
Habia atrimaxillaris_Black-cheeked Ant-Tanager      6
Cantorchilus semibadius_Riverside Wren              3
Habia rubica_Red-crowned Ant-Tanager                2
Cyanoloxia cyanoides_Blue-black Grosbeak            2
Attila spadiceus_Bright-rumped Attila               1
Cantorchilus nigricapillus_Bay Wren                 1
Automolus exsertus_Chiriqui Foliage-gleaner         1
Tringa solitaria_Solitary Sandpiper   

## Observations (Step 3)
- process_all_clips(clips_folder, output_csv) matches the call in run_pipeline.py step3_birdnet (signature now also has use_allow_list=True).
- Input: 1,747 clips from Step 1 (recorder 5). Output: 1,747 rows, 44.9 s, no "Error processing" lines.
- clip_name includes ".wav" and matches Step 1 files and Step 2 output exactly (1,747/1,747).
- species/confidence are stored as strings in the CSV; ast.literal_eval parses them back (equal lengths in all rows).
- 101 of 1,747 clips (5.8%) have detections; 106 detections total. 0 outside the allow-list; min confidence 0.2519 (threshold 0.25).